# Week 43, Day 5: Circular buffers

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w043d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
capacity = 5
slots = [None] * capacity
head, size = 0, 0
for item in "abcdefg":
    if size == capacity:            # full: drop the oldest
        head = (head + 1) % capacity
        size -= 1
    slots[(head + size) % capacity] = item
    size += 1
print(slots, head, size)
print([slots[(head + i) % capacity] for i in range(size)])

In [ ]:
class RingQueue:
    def __init__(self, capacity):
        self._slots = [None] * capacity
        self._head = 0                         # slot of the oldest item
        self._size = 0

    def enqueue(self, item):
        if self._size == len(self._slots):
            raise OverflowError("queue is full")
        self._slots[(self._head + self._size) % len(self._slots)] = item
        self._size += 1

    def dequeue(self):
        if self._size == 0:
            raise IndexError("dequeue from empty queue")
        item = self._slots[self._head]
        self._slots[self._head] = None         # let go of the reference
        self._head = (self._head + 1) % len(self._slots)
        self._size -= 1
        return item

q = RingQueue(3)
for x in [1, 2, 3]:
    q.enqueue(x)
print(q.dequeue(), q.dequeue())
q.enqueue(4)
q.enqueue(5)
print(q._slots, q._head, [q.dequeue() for _ in range(3)])

In [ ]:
class GrowingQueue:
    def __init__(self):
        self._slots = [None]
        self._head = self._size = 0

    def enqueue(self, item):
        if self._size == len(self._slots):
            old, cap = self._slots, len(self._slots)
            self._slots = [old[(self._head + i) % cap] for i in range(self._size)] + [None] * cap
            self._head = 0                         # the oldest is now in slot 0
        self._slots[(self._head + self._size) % len(self._slots)] = item
        self._size += 1

    def dequeue(self):
        item = self._slots[self._head]
        self._slots[self._head] = None
        self._head = (self._head + 1) % len(self._slots)
        self._size -= 1
        return item

q = GrowingQueue()
order = []
for i in range(10):
    q.enqueue(i)
    if i % 3 == 2:
        order.append(q.dequeue())
while q._size:
    order.append(q.dequeue())
print(order, len(q._slots))

> **Key idea.** A circular buffer is a fixed block with a head index and a size: item `i` is in slot `(head + i) % capacity`, the next free slot is `(head + size) % capacity`, and every operation is O(1) in the worst case, with nothing shifted or allocated. With only head and tail indexes, `head == tail` can mean full or empty: keep a size (or one empty slot) to tell them apart. When full, a ring can refuse, overwrite the oldest, or grow by copying its items in order, oldest first, into a bigger block. Stepping the head back with `(head - 1) % capacity` adds at the front, making a deque.

### Exercise: A ring queue

Write `RingQueue(capacity)` on a fixed block `[None] * capacity` with a head index and a size. It needs `enqueue` (raising `OverflowError("queue is full")`), `dequeue` and `peek` (raising `IndexError("dequeue from empty queue")` and `IndexError("peek at empty queue")`), `__len__`, `is_full()`, `__iter__` (oldest first, without `yield`), and `__repr__` like `RingQueue([3, 4], capacity=3)`. `dequeue` sets its freed slot back to `None`. A capacity below 1 raises `ValueError`.

In [ ]:
class RingQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef test_ring():\n    \"a ring\"\n    q = RingQueue(3)\n    for x in [1, 2, 3]:\n        q.enqueue(x)\n    assert q.is_full() and len(q) == 3\n    try:\n        q.enqueue(4)\n        assert False\n    except OverflowError as err:\n        assert str(err) == \"queue is full\"\n    assert q.dequeue() == 1 and q.dequeue() == 2\n    q.enqueue(4)\n    q.enqueue(5)\n    assert list(q) == [3, 4, 5] and repr(q) == \"RingQueue([3, 4, 5], capacity=3)\" and q.peek() == 3\n    assert q._slots.count(None) == 0 and len(q._slots) == 3\n    assert [q.dequeue() for _ in range(3)] == [3, 4, 5] and q._slots == [None, None, None] and len(q) == 0\n    for name, message in [(\"dequeue\", \"dequeue from empty queue\"), (\"peek\", \"peek at empty queue\")]:\n        try:\n            getattr(q, name)()\n            assert False\n        except IndexError as err:\n            assert str(err) == message\n    try:\n        RingQueue(0)\n        assert False\n    except ValueError:\n        pass\ndef test_fixed():\n    \"a fixed block\"\n    q = RingQueue(4)\n    for i in range(10_000):\n        q.enqueue(i)\n        assert q.dequeue() == i\n    assert len(q._slots) == 4\n    calls = [n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.Call) and getattr(n.func, \"attr\", \"\") in (\"append\", \"pop\", \"insert\")]\n    assert not calls and \"yield\" not in source, \"index into the block; no list methods\"")

### Exercise: A log that keeps the latest

Write `RingLog(capacity)`: `add(entry)` stores an entry, overwriting the **oldest** when full; `entries()` returns them oldest first; `latest(n)` returns the newest `n` (or all, if there are fewer), newest first; `len` is how many are kept. Use a fixed block and a head index; don't use `deque`.

In [ ]:
class RingLog:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_log():\n    \"the latest entries\"\n    log = RingLog(3)\n    for e in \"abcde\":\n        log.add(e)\n    assert log.entries() == [\"c\", \"d\", \"e\"] and len(log) == 3 and log.latest(2) == [\"e\", \"d\"] and log.latest(10) == [\"e\", \"d\", \"c\"]\n    small = RingLog(5)\n    small.add(\"x\")\n    assert small.entries() == [\"x\"] and small.latest(3) == [\"x\"] and len(small._slots) == 5\n    assert \"deque\" not in source")

### Exercise: A ring that grows

Write `GrowingQueue()`: a ring queue starting at capacity 1 that **doubles** when full, copying its items oldest first into the new block (with the oldest in slot 0). It needs `enqueue`, `dequeue` (raising `IndexError` when empty), `__len__` and a read-only property `capacity`. The checks mix enqueues and dequeues so the ring has wrapped before it grows.

In [ ]:
class GrowingQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef test_grow():\n    \"against a deque\"\n    rng = random.Random(1)\n    q, ref = GrowingQueue(), deque()\n    for i in range(3000):\n        if ref and rng.random() < 0.4:\n            assert q.dequeue() == ref.popleft()\n        else:\n            q.enqueue(i)\n            ref.append(i)\n        assert len(q) == len(ref)\n    while ref:\n        assert q.dequeue() == ref.popleft()\n    try:\n        q.dequeue()\n        assert False\n    except IndexError:\n        pass\ndef test_capacity():\n    \"doubling\"\n    q = GrowingQueue()\n    caps = []\n    for i in range(9):\n        q.enqueue(i)\n        caps.append(q.capacity)\n    assert caps == [1, 2, 4, 4, 8, 8, 8, 8, 16]\n    try:\n        q.capacity = 3\n        assert False\n    except AttributeError:\n        pass")

### Exercise: A deque on a ring

Write `RingDeque(capacity)` on a fixed block: `push_front`, `push_back` (both raising `OverflowError` when full), `pop_front`, `pop_back` (both raising `IndexError` when empty), `__len__` and `items()` (front to back). Adding at the front steps the head back with `(head - 1) % capacity`.

In [ ]:
class RingDeque:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef test_deque():\n    \"both ends\"\n    d = RingDeque(4)\n    d.push_back(2); d.push_front(1); d.push_back(3); d.push_front(0)\n    assert d.items() == [0, 1, 2, 3] and len(d) == 4\n    try:\n        d.push_back(9)\n        assert False\n    except OverflowError:\n        pass\n    assert d.pop_back() == 3 and d.pop_front() == 0 and d.items() == [1, 2]\n    rng = random.Random(2)\n    d, ref = RingDeque(6), deque()\n    for i in range(2000):\n        op = rng.choice([\"pf\", \"pb\", \"of\", \"ob\"])\n        if op in (\"pf\", \"pb\") and len(ref) < 6:\n            (d.push_front if op == \"pf\" else d.push_back)(i)\n            (ref.appendleft if op == \"pf\" else ref.append)(i)\n        elif op in (\"of\", \"ob\") and ref:\n            assert (d.pop_front() if op == \"of\" else d.pop_back()) == (ref.popleft() if op == \"of\" else ref.pop())\n        assert d.items() == list(ref)\n    assert len(d._slots) == 6")

### Exercise: Average of the latest readings

Write `LatestAverage(n)` with `add(x)`, which stores a reading and returns the average of the latest `n` readings (or of all of them, if there are fewer than `n` so far), in O(1): keep the readings in a ring and a running total, subtracting a reading when it's overwritten.

In [ ]:
class LatestAverage:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_avg():\n    \"averages\"\n    a = LatestAverage(3)\n    assert [a.add(x) for x in [3, 6, 9, 12, 3]] == [3.0, 4.5, 6.0, 9.0, 8.0]\n    rng = random.Random(3)\n    xs = [rng.randint(0, 100) for _ in range(500)]\n    a = LatestAverage(7)\n    for i, x in enumerate(xs):\n        want = sum(xs[max(0, i - 6):i + 1]) / min(i + 1, 7)\n        assert abs(a.add(x) - want) < 1e-9\ndef test_fast():\n    \"O(1)\"\n    a = LatestAverage(10_000)\n    for x in range(200_000):\n        last = a.add(x)\n    assert last == sum(range(190_000, 200_000)) / 10_000")

### Bug hunt: Off the end of the block

`enqueue` works out the next free slot as `head + size` without wrapping, so once the ring has moved round it writes past the end of the block and crashes with `IndexError`. Fix it so this prints `[3, 4, 5]`.

In [ ]:
slots = [None] * 3
head, size = 0, 0

def enqueue(item):
    global size
    slots[head + size] = item
    size += 1

def dequeue():
    global head, size
    item = slots[head]
    head = (head + 1) % len(slots)
    size -= 1
    return item

for x in [1, 2, 3]:
    enqueue(x)
dequeue(); dequeue()
enqueue(4); enqueue(5)
print([slots[(head + i) % 3] for i in range(size)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"wrap with %\"\n    expect_output(\"[3, 4, 5]\")")

### Bug hunt: Full looks empty

This ring keeps a head and a tail and no size, and decides it's empty when `head == tail`. But a **full** ring also has `head == tail`, so after filling it, it claims to be empty: this prints `True`. Add a size count, and use it for both tests, so this prints `False 3`.

In [ ]:
class Ring:
    def __init__(self, capacity):
        self.slots = [None] * capacity
        self.head = self.tail = 0

    def push(self, item):
        self.slots[self.tail] = item
        self.tail = (self.tail + 1) % len(self.slots)

    def is_empty(self):
        return self.head == self.tail

r = Ring(3)
for x in "abc":
    r.push(x)
print(r.is_empty())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tell full from empty\"\n    expect_output(\"False 3\")")

### Bug hunt: Grown out of order

When this ring grows, it copies the slots in their **physical** order, slot 0 first, but after wrapping the oldest item isn't in slot 0, so the queue comes out jumbled: this prints `[2, 0, 1, 3]`. Copy oldest first, `(head + i) % cap`, so it prints `[0, 1, 2, 3]`.

In [ ]:
slots, head, size = [None] * 3, 0, 0

def enqueue(item):
    global slots, head, size
    cap = len(slots)
    if size == cap:
        slots = slots + [None] * cap
        head = 0
        cap *= 2
    slots[(head + size) % cap] = item
    size += 1

def dequeue():
    global head, size
    item = slots[head]
    head = (head + 1) % len(slots)
    size -= 1
    return item

enqueue("x"); enqueue(0); dequeue()
enqueue(1); enqueue(2)            # wraps: 2 goes into slot 0
enqueue(3)                        # full: grows
print([dequeue() for _ in range(4)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"oldest first\"\n    expect_output(\"[0, 1, 2, 3]\")")

### Puzzle: Dequeue from a ring

Arrange the lines into a ring queue's `dequeue`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
capacity, head, size = 5, 3, 4

a = (head + size) % capacity
b = [(head + i) % capacity for i in range(size)]
c = (0 - 1) % capacity
d = (head + size - 1) % capacity

p = ___   # a: the next free slot
q = ___   # b: the slots holding items, oldest first
r = ___   # c
s = ___   # d: the slot of the newest item
t = ___   # the worst-case cost of a fixed ring's enqueue: "O(1)" or "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == [3, 4, 0, 1] and r == 4, \"p, q and r\"\n    assert s == 1 and t == \"O(1)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a hit counter in fixed memory

A website wants to know how many requests arrived in the last `window` seconds, using memory that **doesn't grow** with traffic: millions of requests a second must still fit. Write `HitCounter(window)` with `hit(t)` (a request at whole second `t`; times never go backwards) and `count(t)` (requests at times `t - window + 1` to `t`; `t` is never before the latest hit). Use two fixed rings of `window` slots, indexed by `t % window`: one holds the **second** each slot was last used for, the other the **number** of hits in that second. When a slot is reused for a newer second, reset its count. Both methods must be O(window) at worst, and `hit` O(1). The checks make a million hits.

In [ ]:
class HitCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import Counter\ndef test_counts():\n    \"against a simple version\"\n    rng = random.Random(4)\n    for window in [1, 5, 60]:\n        h = HitCounter(window)\n        hits, t = [], 0\n        for _ in range(500):\n            t += rng.choice([0, 0, 1, 2, 7])\n            if rng.random() < 0.7:\n                h.hit(t)\n                hits.append(t)\n            assert h.count(t) == sum(1 for x in hits if t - window < x <= t), (window, t)\n        assert h.count(t + window) == 0\ndef test_memory():\n    \"fixed memory\"\n    h = HitCounter(300)\n    for i in range(1_000_000):\n        h.hit(i // 1000)\n    assert h.count(999) == 300_000 and h.count(1100) == 199_000\n    assert all(len(v) == 300 for v in vars(h).values() if isinstance(v, list)), \"only fixed rings\"")

## Warm-up

Seven items went into five slots. The last two **wrapped round** and overwrote the oldest, `a` and `b`. The block itself says `['f', 'g', 'c', 'd', 'e']`, but reading from `head`, wrapping as you go, gives the items oldest first: `c d e f g`.

## Wrapping round

A circular buffer is a fixed block of `capacity` slots, used as if its end joined back on to its start:

```text
 capacity 6, head = 4, size = 4

 slot:    0     1     2     3     4     5
        +-----+-----+-----+-----+-----+-----+
        |  c  |  d  |     |     |  a  |  b  |
        +-----+-----+-----+-----+-----+-----+
                       ^           ^
                    next free     head (the oldest)

 item i (oldest first) is in slot (head + i) % capacity
 the next free slot is (head + size) % capacity
```

The `% capacity` (Week 2's remainder) is the wrap: index 6 becomes 0, 7 becomes 1. With that, the queue operations are each a couple of lines:

Every operation is **O(1) in the worst case** (not just amortised), touches one slot, and creates no objects. That's why circular buffers are used where timing must be predictable: audio and video playback, the keyboard buffer, data arriving from a network card.

## Full or empty?

There's a classic trap. Some ring buffers keep a **head** and a **tail** index and no size. Then `head == tail` happens in two situations: when the buffer is **empty**, and when it's **completely full** (the tail has wrapped all the way round to meet the head). Three ways out:

| Method | How | Cost |
|---|---|---|
| keep a **size** count | full is `size == capacity`, empty is `size == 0` | one extra number |
| keep one slot **always empty** | full is `(tail + 1) % capacity == head` | one wasted slot |
| keep a **full flag** | set when the tail catches the head | one extra flag, and more cases |

This course keeps a size, which makes `len` free too. Low-level code often prefers the empty-slot method, because it needs no shared counter when one part of a program only adds and another only removes.

## When it's full

What should happen when an item arrives and there's no room? It depends on what the buffer is for:

| Policy | Behaviour | Used for |
|---|---|---|
| **refuse** | raise an error, or tell the producer to wait | job queues where nothing may be lost |
| **overwrite** | drop the oldest item to make room | logs of recent events, the last few seconds of audio or sensor data |
| **grow** | move to a bigger block | general-purpose queues (Python's `deque` grows block by block) |

Overwriting is what `deque(maxlen=n)` does (Day 2). It's the right choice when only the **most recent** data matters, and the wrong one when every item is precious, such as orders in a shop.

## Growing a ring

To grow, make a bigger block and copy the items across **in order**, oldest first, starting from slot 0 of the new block. Copying the slots in their physical order would be wrong, because after wrapping, the oldest item isn't in slot 0:

Doubling gives O(1) amortised enqueues, for the same reason as Week 41's dynamic array.

## Both ends, and uses

A ring can work at **both** ends: to add at the front, step the head **back** one slot, `(head - 1) % capacity` (in Python, `-1 % 5` is 4, so this wraps correctly), and write there. That's a deque on a fixed block.

Rings are also handy for **running statistics over the latest n values**: keep the last n readings in a ring and a running total; each new reading subtracts the one it overwrites and adds itself, so the average of the latest n is O(1) per reading, with no window to slide over a list (Week 41's sliding window, as a structure).

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a ring of 5 slots on paper and run the warm-up by hand, marking the head after each item. Then decide, for three programs you use, whether a full buffer should refuse, overwrite or grow. Tomorrow is the Forge: a print shop's job system built from this week's stacks, queues and rings.